# Lux API example ⚡️

This notebook demonstrates the high-level `LuxModel` API (`src/lux_model.py`), which wraps
the lower-level routines (`init_latents`, `optimise`, `scatters`) into a single model object,
on APOGEE giant spectra with asteroseismic ages:

1. load and clean the spectra + labels,
2. continuum-normalise the fluxes and propagate the uncertainties,
3. **train** the model latents (`fit`),
4. **predict labels from spectra** for held-out stars (`predict_labels`),
5. **predict spectra from labels** (`predict_fluxes`),
6. inspect the latent space, and
7. save / load the trained model.

In [ ]:
import sys, os

# make src importable both locally and on Gadi
# (sys.path entries must be strings, not pathlib.Path objects)
for cand in ['../src', '/scratch/mk27/mj8805/Lux/src']:
    if os.path.exists(os.path.join(cand, 'lux_model.py')):
        sys.path.append(os.path.abspath(cand))
        break
else:
    raise FileNotFoundError('could not locate Lux/src — adjust the candidate paths above')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

from lux_model import LuxModel

plt.rcParams['figure.dpi'] = 110

In [ ]:
import pandas as pd
from pathlib import Path
SRC = Path('/home/100/mj8805/scr_mk27/Lux')

spectra = pd.read_parquet(SRC / '..' / 'bulge-ages-and-orbits' / 'data' / 'merged_with_ages.parquet')
spectra.head()

In [ ]:
# Remove rows where spectra_flags != 0
spectra = spectra[spectra['spectrum_flags'] == 0]

# Find all columns ending with '_warn'
warn_cols = [col for col in spectra.columns if col.endswith('_warn')]

# Remove rows where any _warn column is True
if warn_cols:
    mask = ~(spectra[warn_cols].any(axis=1))
    spectra = spectra[mask].reset_index(drop=True)

In [ ]:
# Remove sentinel / non-physical label values BEFORE computing the clipping statistics.
# NOTE: [Fe/H] is legitimately negative, so only -9999-style sentinels are removed there;
# Teff, logg, and age must be physically positive.
for lbl in ['raw_teff', 'raw_logg', 'age_Dnu']:
    spectra = spectra[spectra[lbl] > 0].reset_index(drop=True)
spectra = spectra[spectra['raw_fe_h'] > -10].reset_index(drop=True)

# Sigma clip by labels (remove rows with any label farther than 3 sigma from the median)
label_names_for_clip = ['raw_teff', 'raw_logg', 'raw_fe_h', 'age_Dnu']

for lbl in label_names_for_clip:
    values = spectra[lbl]
    if np.all(np.isnan(values)) or len(values.dropna()) <= 1:
        continue
    med = np.nanmedian(values)
    std = np.nanstd(values)
    if std == 0 or np.isnan(std):
        continue
    mask = np.abs(values - med) < 3 * std
    spectra = spectra[mask].reset_index(drop=True)

print(f'{len(spectra)} stars after cleaning')

In [ ]:
# Continuum-normalise the spectra and propagate the uncertainties.
# For f_norm = f / c the error is sigma_norm = sigma / c = 1 / (sqrt(ivar) * c),
# where ivar is the inverse variance of the *un-normalised* flux.
raw_flux = np.array(spectra['flux'].tolist())
raw_ivar = np.array(spectra['ivar'].tolist())
cont = np.array(spectra['continuum'].tolist())
wl = jnp.array(np.array(spectra['wavelength'].tolist()))

# bad pixels: non-positive flux, missing continuum, or zero ivar (e.g. chip edges)
bad = (raw_flux <= 0) | (cont <= 0) | (raw_ivar <= 0)
safe_cont = np.where(cont > 0, cont, 1.)
safe_ivar = np.where(raw_ivar > 0, raw_ivar, 1.)

# same sentinel convention as load_data.load_spectra: flux=1, err=9999 for bad pixels
fluxes = jnp.array(np.where(bad, 1., raw_flux / safe_cont))
fluxes_err = jnp.array(np.where(bad, 9999., 1. / (np.sqrt(safe_ivar) * safe_cont)))

label_names = ['raw_teff', 'raw_logg', 'raw_fe_h', 'age_Dnu']
label_errs = ['raw_e_teff', 'raw_e_logg', 'raw_e_fe_h', 'e_age_Dnu']

spectra['e_age_Dnu'] = (spectra['age_84_Dnu'] - spectra['age_16_Dnu']) / 2.

labels = jnp.array(spectra[label_names].values)
labels_err_np = np.array(spectra[label_errs].values, dtype=float)
labels_err = jnp.array(np.where(labels_err_np > 0, labels_err_np, 9999.))

print(f'{fluxes.shape[0]} stars, {fluxes.shape[1]} pixels, '
      f'bad pixel fraction: {bad.mean():.3f}')

In [ ]:
star0_good = np.asarray(fluxes_err[0] < 100)  # hide the 9999-sentinel pixels

plt.figure(figsize=(9, 3.5))
plt.plot(wl[0], fluxes[0], lw=0.7, label='flux')
plt.fill_between(np.asarray(wl[0]),
                 np.asarray(fluxes[0] - fluxes_err[0]),
                 np.asarray(fluxes[0] + fluxes_err[0]),
                 where=star0_good, color='gray', alpha=0.4, label='flux error')
plt.ylim(0.5, 1.3)
plt.xlabel('wavelength [$\AA$]')
plt.ylabel('normalised flux')
plt.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, len(label_names), figsize=(16, 8))

for i, name in enumerate(label_names):
    ax = axes[0, i]
    ax.hist(np.asarray(labels[:, i]), bins=30, color='C0', alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel(name)
    ax.set_ylabel('count')

for i, name in enumerate(label_names):
    ax = axes[1, i]
    err = np.asarray(labels_err[:, i])
    ax.hist(err[err < 9000], bins=30, color='C1', alpha=0.7)  # hide 9999 sentinels
    ax.set_title(f'{name} (err)')
    ax.set_xlabel(f'err {name}')
    ax.set_ylabel('count')

plt.tight_layout()
plt.show()

## 2. Train the model

`fit` initialises the latents (`init_latents`), runs the un-regularised alpha/beta/zeta
coordinate-descent agenda (`optimise.run_agenda`) for `n_iterations`, then by default the
regularised agenda that also fits per-pixel noise scatters (`scatters.run_agenda`).

The latent dimensionality must satisfy `P >= M + 1` (one continuum dimension plus one per
label); with M = 4 labels the minimum is `P = 5` used here. To choose `P` properly,
cross-validate — see `kfold_cv.py` / `run_kfold.py`.

In [ ]:
# Create a training mask
from sklearn.model_selection import train_test_split

n_samples = labels.shape[0]
train_mask, test_mask = train_test_split(np.arange(n_samples), test_size=0.2, random_state=42)

model = LuxModel(P=5)
model.fit(labels[train_mask], labels_err[train_mask], fluxes[train_mask], fluxes_err[train_mask],
          n_iterations=5, l2_reg_strength=1.0, label_names=label_names)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, len(model.chi2_history) + 1), model.chi2_history, 'o-')
ax.set_xlabel('iteration')
ax.set_ylabel('$\chi^2$')
ax.set_title('coordinate-descent convergence')
plt.tight_layout()

## 3. Predict labels from spectra

For new stars we first infer their latent $\zeta$ from the spectrum alone (holding the
trained betas and pixel scatters fixed), then synthesise labels as $\zeta \alpha^T$.
`predict_labels` does both steps; `return_zetas=True` also returns the latents.

In [ ]:
labels_pred, zetas_test = model.predict_labels(fluxes[test_mask], fluxes_err[test_mask],
                                               return_zetas=True)
labels_pred = np.asarray(labels_pred)

fig, axes = plt.subplots(1, len(label_names), figsize=(14, 3.5))
for m, (ax, name) in enumerate(zip(axes, label_names)):
    truth, pred = np.asarray(labels[test_mask, m]), labels_pred[:, m]
    lims = [truth.min(), truth.max()]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.plot(truth, pred, '.', ms=5)
    bias, scatter = np.mean(pred - truth), np.std(pred - truth)
    ax.set_title(f'{name}: bias={bias:.3g}, scatter={scatter:.3g}', fontsize=9)
    ax.set_xlabel(f'reference {name}')
    ax.set_ylabel(f'predicted {name}')
plt.tight_layout()

The prediction scatter should be comparable to the quoted label errors. Keep in mind the
scatter is measured against the *noisy* reference labels, so it bundles the model's
prediction error with the reference-label noise itself.

## 4. Predict spectra from labels

The reverse direction: infer $\zeta$ from the labels alone, then synthesise the spectrum
as $\zeta \beta^T$. With `P > M` this inference is under-determined, so by default
`predict_fluxes` uses a MAP estimate with an empirical Gaussian prior built from the
training zetas, which keeps the unconstrained latent directions at typical training values
(`use_prior=False` recovers the original unregularised behaviour).

In [ ]:
fluxes_pred = np.asarray(model.predict_fluxes(labels[test_mask], labels_err[test_mask]))

star = 5  # a test star to look at
obs = np.asarray(fluxes[test_mask][star])
err = np.asarray(fluxes_err[test_mask][star])
good = err < 100

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(wl[0], np.where(good, obs, np.nan), lw=0.8, label='observed')
axes[0].plot(wl[0], fluxes_pred[star], lw=0.8, label='predicted from labels')
axes[0].set_ylabel('normalised flux')
axes[0].legend(fontsize=8)
axes[1].axhline(0., color='k', ls='--', lw=1)
axes[1].plot(wl[0], np.where(good, fluxes_pred[star] - obs, np.nan), lw=0.8)
axes[1].set_xlabel('wavelength [$\AA$]')
axes[1].set_ylabel('residual')
plt.tight_layout()

good_all = np.asarray(fluxes_err[test_mask]) < 100
resid = (fluxes_pred - np.asarray(fluxes[test_mask]))[good_all]
print(f'flux prediction rms over good test pixels: {np.sqrt(np.mean(resid**2)):.4f} '
      f'(median flux error: {np.median(np.asarray(fluxes_err[test_mask])[good_all]):.4f})')

## 5. The latent space

The training zetas are the per-star latent vectors the model has learned. The latent basis
is only defined up to a linear transformation, but label-correlated structure is easy to see —
here we colour two latent dimensions by [Fe/H].

In [ ]:
zetas_train = np.asarray(model.zetas)

fig, ax = plt.subplots(figsize=(5, 4))
sc = ax.scatter(zetas_train[:, 1], zetas_train[:, 2], c=np.asarray(labels[train_mask, 2]),
                s=10, cmap='viridis')
fig.colorbar(sc, ax=ax, label='[Fe/H]')
ax.set_xlabel('$\zeta_1$')
ax.set_ylabel('$\zeta_2$')
ax.set_title('training-set latents')
plt.tight_layout()

## 6. Save and load

`save` stores the hyperparameters, latents, scatters, and label names as a dill pickle;
`LuxModel.load` restores a model that predicts identically.

In [ ]:
path = model.save('lux-example-model.dill')
restored = LuxModel.load(path)

labels_pred_restored = np.asarray(restored.predict_labels(fluxes[test_mask], fluxes_err[test_mask]))
assert np.allclose(labels_pred, labels_pred_restored)
print(f'saved to {path}; restored model predicts identically '
      f'(P={restored.P}, labels={restored.label_names})')